# Custom Authentication with Request Lambda Interceptor

## Introduction

When deploying AI agents with Amazon Bedrock AgentCore, organizations benefit from built-in support for OAuth 2.0, AWS IAM, and API key authentication through Amazon Bedrock AgentCore Gateway. However, some enterprise environments still utilize older authentication mechanisms such as HTTP Basic Authentication (RFC 7617). AgentCore Gateway's extensible architecture makes it straightforward to support these authentication mechanisms through request Lambda interceptors — custom code that runs each time an agent calls a tool.

This tutorial shows how to use a request Lambda interceptor to authenticate to a downstream tool API using system credentials — retrieving a service account credential from AWS Secrets Manager and constructing a Basic Auth header on behalf of the calling agent. This design keeps credentials isolated from the agent runtime, preventing exposure through model-driven behavior such as prompt injection.

**Important**: HTTP Basic Authentication is an antiquated technology that transmits credentials as Base64-encoded text and should not be used as a long-term authentication strategy. AWS recommends modernizing to OAuth 2.0, SAML, or OpenID Connect where possible. However, some organizations with legacy workloads choose to decouple authentication modernization from their agentic AI adoption, addressing each on independent timelines. If your environment requires Basic Auth integration as an interim measure, consult your AWS Solutions Architect to evaluate the security trade-offs before proceeding.

### Tutorial Details

| Information | Details |
| :--- | :--- |
| Tutorial type | Interactive |
| AgentCore components | AgentCore Gateway |
| Gateway Target type | AWS Lambda |
| Inbound Auth IdP | Amazon Cognito (adaptable to any OIDC provider) |
| Outbound Auth | Request Lambda Interceptor (system credential) |
| Example complexity | Intermediate |
| SDK used | boto3, PyJWT |

### Prerequisites

Before running this tutorial, you should have:
- An existing AgentCore Gateway with inbound JWT authentication configured
- A downstream tool API that accepts Basic Auth (service account credential)
- An IAM role for the interceptor Lambda with `secretsmanager:GetSecretValue` and `kms:Decrypt` permissions

**Cost considerations:** This tutorial creates AWS resources that incur charges, including Lambda functions, IAM roles, KMS keys, Secrets Manager secrets, and gateway targets. Follow the Cleanup steps at the end to delete these resources and avoid ongoing costs.

---


## Step 1: Install Dependencies

Install the required packages for interacting with AWS services.

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'boto3'])
print('Dependencies installed successfully')

## Step 2: Setup & Configuration

Initialize AWS clients and set configuration variables. Replace the placeholder values with your own.

In [ ]:
import boto3
import json
import time
import zipfile
import io
import subprocess
import sys
from datetime import datetime
from botocore.exceptions import ClientError

# ── Replace these with your values ──────────────────────────────────────
region = 'us-east-1'                                  # Your AWS region
gateway_id = '<your-gateway-id>'                      # Your existing AgentCore Gateway ID
cognito_user_pool_id = '<your-cognito-user-pool-id>'  # Your Cognito User Pool ID
cognito_region = region                               # Region where Cognito User Pool lives
# ───────────────────────────────────────────────────────────────────────

timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')

lambda_client = boto3.client('lambda', region_name=region)
iam_client = boto3.client('iam', region_name=region)
agentcore_client = boto3.client('bedrock-agentcore-control', region_name=region)
secrets_client = boto3.client('secretsmanager', region_name=region)
kms_client = boto3.client('kms', region_name=region)
sts_client = boto3.client('sts', region_name=region)
account_id = sts_client.get_caller_identity()['Account']

print(f'Region:    {region}')
print(f'Account:   {account_id}')
print(f'Gateway:   {gateway_id}')
print(f'Timestamp: {timestamp}')


## Step 3: Deploy the Request Lambda Interceptor

This is the core of the tutorial. The interceptor Lambda sits between the gateway and your downstream tool. When a request arrives, it:

1. **Validates the inbound JWT** signature, expiration, and issuer (defense-in-depth)
2. **Retrieves the system credential** from Secrets Manager (KMS-encrypted, cached with TTL)
3. **Constructs the Basic Auth header** using the system credential (RFC 7617)
4. Returns the transformed request to the gateway for forwarding

The following cell defines the full interceptor code and deploys it as a Lambda function with the required IAM role, KMS key, Secrets Manager secret, and scoped permissions.


In [ ]:
import os
# ---------------------------------------------------------------------------
# System Credential Interceptor — Lambda Function Code
# ---------------------------------------------------------------------------
interceptor_code = '''
import json
import logging
import base64
import os
import time
import boto3
import jwt
from jwt import PyJWKClient

logger = logging.getLogger()
logger.setLevel(logging.INFO)

secrets_client = boto3.client("secretsmanager")

SYSTEM_CREDS_SECRET_NAME = os.environ["SYSTEM_CREDS_SECRET_NAME"]
COGNITO_USER_POOL_ID = os.environ["COGNITO_USER_POOL_ID"]
COGNITO_REGION = os.environ.get("COGNITO_REGION", "us-east-1")
COGNITO_ISSUER = f"https://cognito-idp.{COGNITO_REGION}.amazonaws.com/{COGNITO_USER_POOL_ID}"
JWKS_URL = f"{COGNITO_ISSUER}/.well-known/jwks.json"
SECRET_CACHE_TTL = int(os.environ.get("SECRET_CACHE_TTL", "300"))

SYSTEM_CREDS = None
SYSTEM_CREDS_LOADED_AT = 0
JWK_CLIENT = None


def get_system_credentials():
    """Retrieve the system service account credential from Secrets Manager (cached with TTL)."""
    global SYSTEM_CREDS, SYSTEM_CREDS_LOADED_AT
    if SYSTEM_CREDS is None or (time.time() - SYSTEM_CREDS_LOADED_AT) > SECRET_CACHE_TTL:
        response = secrets_client.get_secret_value(SecretId=SYSTEM_CREDS_SECRET_NAME)
        SYSTEM_CREDS = json.loads(response["SecretString"])
        SYSTEM_CREDS_LOADED_AT = time.time()
        logger.info("[SystemCreds] System credential loaded from Secrets Manager.")
    return SYSTEM_CREDS


def get_jwk_client():
    """Get or create the JWK client for token validation."""
    global JWK_CLIENT
    if JWK_CLIENT is None:
        JWK_CLIENT = PyJWKClient(JWKS_URL)
    return JWK_CLIENT


def validate_jwt(token):
    """Validate JWT signature, expiration, and issuer (defense-in-depth)."""
    try:
        jwk_client = get_jwk_client()
        signing_key = jwk_client.get_signing_key_from_jwt(token)
        claims = jwt.decode(token, signing_key.key, algorithms=["RS256"], issuer=COGNITO_ISSUER, options={"verify_aud": False})
        logger.info(f"[JWT] Token validated successfully (sub: {claims.get('sub')}).")
        return claims
    except jwt.ExpiredSignatureError:
        logger.error("[JWT] Token has expired.")
        return None
    except jwt.InvalidTokenError as e:
        logger.error(f"[JWT] Token validation failed: {e}")
        return None


def build_system_auth_header(headers):
    """Validate JWT and construct Basic Auth header with system credential."""
    logger.info("[SystemAuth] Processing authentication.")

    auth_header = headers.get("Authorization", "")
    if not auth_header.startswith("Bearer "):
        return _error_response(401, "No Bearer token found in request.")

    claims = validate_jwt(auth_header[7:])
    if not claims:
        return _error_response(401, "JWT validation failed.")

    creds = get_system_credentials()

    basic_auth_encoded = base64.b64encode(
        f"{creds['username']}:{creds['password']}".encode()
    ).decode()
    headers["Authorization"] = f"Basic {basic_auth_encoded}"

    logger.info("[SystemAuth] System credential applied — request authenticated.")
    return headers


def _transformed_request(headers, body):
    return {"interceptorOutputVersion": "1.0", "mcp": {"transformedGatewayRequest": {"headers": headers, "body": body}}}


def _passthrough(gateway_request):
    return {"interceptorOutputVersion": "1.0", "mcp": {"transformedGatewayRequest": {"headers": gateway_request.get("headers", {}), "body": gateway_request.get("body", {})}}}


def _error_response(status_code, message):
    return {"interceptorOutputVersion": "1.0", "mcp": {"transformedGatewayResponse": {"statusCode": status_code, "headers": {}, "body": {"error": message}}}}


def lambda_handler(event, context):
    logger.info("System Credential Interceptor invoked.")

    if "mcp" not in event:
        return _error_response(400, "Invalid request structure.")

    if event["mcp"].get("gatewayResponse") is not None:
        gw_resp = event["mcp"]["gatewayResponse"]
        return {"interceptorOutputVersion": "1.0", "mcp": {"transformedGatewayResponse": {"statusCode": gw_resp.get("statusCode", 200), "headers": gw_resp.get("headers", {}), "body": gw_resp.get("body", {})}}}

    gateway_request = event["mcp"]["gatewayRequest"]
    headers = gateway_request.get("headers", {}).copy()
    body = gateway_request.get("body", {})
    tool_name = body.get("params", {}).get("name", "")
    logger.info(f"[Request] Tool: {tool_name}")

    if "get_user_data" in tool_name.lower() or "basic-auth" in tool_name.lower():
        result = build_system_auth_header(headers)
        if isinstance(result, dict) and "mcp" in result:
            return result
        return _transformed_request(result, body)

    return _passthrough(gateway_request)
'''

# ---------------------------------------------------------------------------
# Create KMS key for encrypting the system credential
# ---------------------------------------------------------------------------
kms_response = kms_client.create_key(
    Description=f'Interceptor system credential encryption key ({timestamp})'
)
kms_key_id = kms_response['KeyMetadata']['KeyId']
kms_key_arn = kms_response['KeyMetadata']['Arn']
print(f'KMS Key created: {kms_key_id}')

# ---------------------------------------------------------------------------
# Create system credential in Secrets Manager (KMS-encrypted)
# ---------------------------------------------------------------------------
secret_name = f'interceptor/system-credentials-{timestamp}'
secrets_client.create_secret(
    Name=secret_name,
    Description='System service account credential for interceptor',
    KmsKeyId=kms_key_id,
    SecretString=json.dumps({
        'username': 'gateway-interceptor-svc',
        'password': 'replace-with-your-service-account-password'
    })
)
print(f'Secret created: {secret_name}')

# ---------------------------------------------------------------------------
# Create IAM role for the Interceptor Lambda
# ---------------------------------------------------------------------------
interceptor_role_name = f'InterceptorLambdaRole-{timestamp}'
interceptor_role_resp = iam_client.create_role(
    RoleName=interceptor_role_name,
    AssumeRolePolicyDocument=json.dumps({
        'Version': '2012-10-17',
        'Statement': [{'Effect': 'Allow', 'Principal': {'Service': 'lambda.amazonaws.com'}, 'Action': 'sts:AssumeRole'}]
    })
)
interceptor_role_arn = interceptor_role_resp['Role']['Arn']

iam_client.attach_role_policy(
    RoleName=interceptor_role_name,
    PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
)

# Secrets Manager + KMS access — scoped to specific resources
iam_client.put_role_policy(
    RoleName=interceptor_role_name, PolicyName='SystemCredsAccess',
    PolicyDocument=json.dumps({
        'Version': '2012-10-17',
        'Statement': [
            {
                'Effect': 'Allow',
                'Action': ['secretsmanager:GetSecretValue'],
                'Resource': [f'arn:aws:secretsmanager:{region}:{account_id}:secret:{secret_name}-*']
            },
            {
                'Effect': 'Allow',
                'Action': ['kms:Decrypt', 'kms:DescribeKey'],
                'Resource': [kms_key_arn]
            }
        ]
    })
)

print('Waiting for IAM role propagation...')
time.sleep(10)

# ---------------------------------------------------------------------------
# Package and deploy the Lambda function (with PyJWT + cryptography)
# ---------------------------------------------------------------------------
# Install PyJWT with crypto dependencies for Lambda packaging
import tempfile, shutil
pkg_dir = tempfile.mkdtemp()
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'PyJWT[crypto]',
     '--platform', 'manylinux2014_x86_64', '--target', pkg_dir,
     '--implementation', 'cp', '--python-version', '3.13',
     '--only-binary=:all:', '--quiet'],
    check=True
)

zip_buf = io.BytesIO()
with zipfile.ZipFile(zip_buf, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('lambda_function.py', interceptor_code)
    # Add PyJWT and dependencies
    for root, dirs, files in os.walk(pkg_dir):
        for file in files:
            if '__pycache__' in root or '.dist-info' in root:
                continue
            full_path = os.path.join(root, file)
            arc_name = os.path.relpath(full_path, pkg_dir)
            zf.write(full_path, arc_name)
zip_buf.seek(0)
shutil.rmtree(pkg_dir)

interceptor_fn_name = f'SystemCredInterceptor-{timestamp}'
interceptor_resp = lambda_client.create_function(
    FunctionName=interceptor_fn_name,
    Runtime='python3.13',
    Role=interceptor_role_arn,
    Handler='lambda_function.lambda_handler',
    Code={'ZipFile': zip_buf.read()},
    Environment={'Variables': {
        'SYSTEM_CREDS_SECRET_NAME': secret_name,
        'COGNITO_USER_POOL_ID': cognito_user_pool_id,
        'COGNITO_REGION': cognito_region,
        'SECRET_CACHE_TTL': '300'
    }},
    Timeout=30,
    MemorySize=256
)
interceptor_arn = interceptor_resp['FunctionArn']
print(f'Interceptor Lambda created: {interceptor_arn}')

# Add resource policy to restrict invocation to the gateway only
lambda_client.add_permission(
    FunctionName=interceptor_fn_name,
    StatementId='AllowGatewayInvoke',
    Action='lambda:InvokeFunction',
    Principal='bedrock-agentcore.amazonaws.com',
    SourceArn=f'arn:aws:bedrock-agentcore:{region}:{account_id}:gateway/{gateway_id}'
)
print('Lambda resource policy added: restricted to gateway invocation only')

# Wait for Active state
for _ in range(12):
    time.sleep(5)
    state = lambda_client.get_function(FunctionName=interceptor_fn_name)['Configuration']['State']
    if state == 'Active':
        print('Interceptor Lambda is active.')
        break
else:
    print(f'Warning: Lambda state is {state}')


## Step 4: Attach Interceptor to Gateway and Configure Target

This step attaches the interceptor to your gateway and creates a gateway target.

**Attaching the interceptor:** You must enable `passRequestHeaders`. Without it, the interceptor cannot receive the `Authorization` header containing the inbound JWT, and JWT validation will not work.

**Gateway target:** The interceptor handles all downstream authentication, so the target does not require gateway-level credential configuration.


In [ ]:
# ---------------------------------------------------------------------------
# Attach interceptor to gateway
# ---------------------------------------------------------------------------
agentcore_client.update_gateway(
    gatewayIdentifier=gateway_id,
    interceptorConfigurations=[
        {
            'interceptor': {
                'lambda': {
                    'arn': interceptor_arn
                }
            },
            'interceptionPoints': ['REQUEST'],
            'inputConfiguration': {
                'passRequestHeaders': True
            }
        }
    ]
)

print('Waiting for gateway update...')
for _ in range(30):
    time.sleep(10)
    status = agentcore_client.get_gateway(gatewayIdentifier=gateway_id)['status']
    print(f'  Status: {status}')
    if status == 'READY':
        print('Interceptor attached successfully.')
        break

# ---------------------------------------------------------------------------
# Create Gateway Target (No Auth — interceptor handles authentication)
# ---------------------------------------------------------------------------
# Replace <your-api-endpoint> with your downstream tool API endpoint.
openapi_spec = {
    'openapi': '3.0.0',
    'info': {'title': 'Enterprise Tool API', 'version': '1.0.0'},
    'servers': [{'url': 'https://<your-api-endpoint>.execute-api.<region>.amazonaws.com/prod'}],
    'paths': {
        '/data': {
            'post': {
                'summary': 'Get data from tool API',
                'operationId': 'get_user_data',
                'responses': {'200': {'description': 'Success', 'content': {'application/json': {'schema': {'type': 'object'}}}}}
            }
        }
    }
}

target_resp = agentcore_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name=f'EnterpriseToolTarget-{timestamp}',
    description='Enterprise tool with Basic Auth (interceptor handles authentication)',
    targetConfiguration={
        'mcp': {
            'openApiSchema': {
                'inlinePayload': json.dumps(openapi_spec)
            }
        }
    }
)
target_id = target_resp['targetId']
print(f'\nGateway target created: {target_id} (No Auth — interceptor handles credentials)')

# Wait for target to be ready
for _ in range(12):
    time.sleep(5)
    t_status = agentcore_client.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)['status']
    if t_status == 'READY':
        print('Target is ready.')
        break


## Step 5: Verify Authentication Pattern

To verify the interceptor is working correctly:

1. Invoke the gateway with a valid JWT token. Check CloudWatch Logs for the interceptor Lambda to confirm the following log sequence:
   - `[JWT] Token validated successfully` — JWT signature verification passed
   - `[SystemCreds] System credential loaded from Secrets Manager` — credential retrieved
   - `[SystemAuth] System credential applied — request authenticated` — Basic Auth header constructed

If the interceptor encounters errors, the logs will show the specific failure point (e.g., `[JWT] Token has expired`, `[JWT] Token validation failed`, `[SystemCreds] Failed to retrieve system credential`).


## Summary and Resource Information

### Resources

* [Gateway Request Interceptor — AWS Documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-interceptors.html)
* [AWS Secrets Manager — Developer Guide](https://docs.aws.amazon.com/secretsmanager/latest/userguide/intro.html)
* [Rotate Active Directory credentials stored in AWS Secrets Manager](https://aws.amazon.com/blogs/modernizing-with-aws/rotate-active-directory-credentials-stored-in-aws-secrets-manager/)
* [Header Propagation with Gateway — AWS Documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-headers.html)


In [ ]:
print('=' * 80)
print('RESOURCE SUMMARY')
print('=' * 80)
print(f'\nInterceptor Lambda:  {interceptor_arn}')
print(f'Interceptor Role:    {interceptor_role_arn}')
print(f'Gateway ID:          {gateway_id}')
print(f'Target ID:           {target_id}')
print(f'KMS Key:             {kms_key_id}')
print(f'Secret Name:         {secret_name}')
print('\n' + '=' * 80)


## Cleanup

Delete the resources created during this tutorial to avoid ongoing charges. Run the cells below in order.

In [ ]:
print('Starting cleanup...\n')

# 1. Delete Gateway Target
try:
    agentcore_client.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
    print(f'Deleted target: {target_id}')
except ClientError as e:
    print(f'Target: {e}')
time.sleep(10)

# 2. Remove interceptor from gateway
try:
    agentcore_client.update_gateway(
        gatewayIdentifier=gateway_id,
        interceptorConfigurations=[]
    )
    print('Interceptor detached from gateway')
except ClientError as e:
    print(f'Gateway update: {e}')
time.sleep(10)

# 3. Delete Interceptor Lambda
try:
    lambda_client.delete_function(FunctionName=interceptor_fn_name)
    print(f'Deleted Lambda: {interceptor_fn_name}')
except ClientError as e:
    print(f'Lambda: {e}')

# 4. Delete IAM Role
try:
    iam_client.delete_role_policy(RoleName=interceptor_role_name, PolicyName='SystemCredsAccess')
    iam_client.detach_role_policy(RoleName=interceptor_role_name,
        PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole')
    iam_client.delete_role(RoleName=interceptor_role_name)
    print(f'Deleted role: {interceptor_role_name}')
except ClientError as e:
    print(f'IAM: {e}')

# 5. Delete Secret
try:
    secrets_client.delete_secret(SecretId=secret_name, ForceDeleteWithoutRecovery=True)
    print(f'Deleted secret: {secret_name}')
except ClientError as e:
    print(f'Secret: {e}')

# 6. Schedule KMS key deletion (minimum 7-day waiting period)
try:
    kms_client.schedule_key_deletion(KeyId=kms_key_id, PendingWindowInDays=7)
    print(f'KMS key scheduled for deletion: {kms_key_id}')
except ClientError as e:
    print(f'KMS: {e}')

print('\nCleanup completed!')


## Conclusion

A request Lambda interceptor in AgentCore Gateway can bridge the gap between the authentication patterns supported by Gateway and the authentication requirements of legacy tool APIs that have not yet migrated to modern authentication standards. As demonstrated in this tutorial, the interceptor validates the inbound JWT, retrieves system credentials from Secrets Manager, and constructs the downstream tool's Basic Auth header — without modifying tool schemas or agent implementation.

This approach is an interim integration pattern, not a target architecture. It introduces a credential that must be synchronized between Secrets Manager and the tool's identity store (such as Active Directory), adding operational overhead for rotation, drift detection, and lifecycle management. The recommended path is to modernize the downstream tool to accept OAuth 2.0, SAML, or OpenID Connect — eliminating stored credentials entirely. Until that modernization is complete, the interceptor isolates credential handling from the agent runtime, ensuring that the agent — a non-deterministic system influenced by user prompts — never has access to authentication secrets.
